In [ ]:
import os
from pathlib import Path

import matplotlib as mpl
import numpy as np
import pandas as pd
import xarray as xr

In [ ]:
ds = xr.open_dataset(os.path.join(Path.home(), "FjordsSim_data", "oslofjord", "OF_inner_105to232_forcing.nc"))
ds_grid = xr.open_dataset(
    os.path.join(Path.home(), "FjordsSim_data", "oslofjord", "OF_inner_105to232_bathymetry_v3.nc")
)

In [ ]:
np_z_centers = (ds_grid.z_faces.values[:-1] + ds_grid.z_faces.values[1:]) / 2
oy, ox, oz = (value for value in ds_grid.sizes.values())
np_mask = (np_z_centers > ds_grid.h.values[..., np.newaxis]).astype(int).transpose(2, 0, 1)

In [ ]:
np_mask_u = np.zeros((oz - 1, oy, ox + 1))
np_mask_u[:, :, :-1] = np_mask
np_mask_u[:, :, 1:] = np.where(np_mask_u[:, :, 1:] == 0, np_mask, np_mask_u[:, :, 1:])

In [ ]:
np_mask_v = np.zeros((oz - 1, oy + 1, ox))
np_mask_v[:, :-1, :] = np_mask
np_mask_v[:, 1:, :] = np.where(np_mask_v[:, 1:, :] == 0, np_mask, np_mask_v[:, 1:, :])

In [ ]:
new_time = pd.date_range(start="2024-01-01T12:00:00", end="2024-12-31T12:00:00", freq="D")

In [ ]:
new_time[~new_time.isin(ds.time.values)]

In [ ]:
ds_new = ds.reindex(time=new_time, method="ffill")

The southern boundary forcing

In [ ]:
nothern_edge = 160

In [ ]:
ds_new["T_lambda"].values[:] = 0
ds_new["T_lambda"][:, :, :nothern_edge, :] = (1 / (60 * 60 * 24)) * np_mask[:, :nothern_edge, :]
ds_new["T_lambda"] = ds_new["T_lambda"].astype(np.float32)

In [ ]:
ds_new["S_lambda"].values[:] = 0
ds_new["S_lambda"][:, :, :nothern_edge, :] = (1 / (60 * 60 * 24)) * np_mask[:, :nothern_edge, :]
ds_new["S_lambda"] = ds_new["S_lambda"].astype(np.float32)

In [ ]:
def fill_bgh_var(df, da, repeat=7):
    arr = df.values[:, 1:]
    arr = arr[:, ::-1]
    first_col = arr[:, [0]]
    extra_cols = np.repeat(first_col, repeat, axis=1)
    arr = np.hstack((extra_cols, arr))  # add bottom layers
    arr = np.vstack((arr, arr[[-1], :]))  # add the last day
    arr = arr[:, :, None, None]
    arr = np.tile(arr, (1, 1, da.shape[2], da.shape[3]))
    return xr.DataArray(arr, dims=da.dims, coords=da.coords, attrs=da.attrs).astype(np.float32)

In [ ]:
df_DOM = pd.read_csv(os.path.join(Path.home(), "FjordsSim_data", "oslofjord", "DOM.txt"), sep="\t")
ds_new["DOM"] = fill_bgh_var(df_DOM, ds_new["S"])
ds_new["DOM_lambda"] = ds_new["S_lambda"]

In [ ]:
df_NUT = pd.read_csv(os.path.join(Path.home(), "FjordsSim_data", "oslofjord", "NUT.txt"), sep="\t")
ds_new["NUT"] = fill_bgh_var(df_NUT, ds_new["S"])
ds_new["NUT_lambda"] = ds_new["S_lambda"]

In [ ]:
df_O2 = pd.read_csv(os.path.join(Path.home(), "FjordsSim_data", "oslofjord", "O2.txt"), sep="\t")
ds_new["O₂"] = fill_bgh_var(df_O2, ds_new["S"])
ds_new["O₂_lambda"] = ds_new["S_lambda"]

In [ ]:
df_P = pd.read_csv(os.path.join(Path.home(), "FjordsSim_data", "oslofjord", "P.txt"), sep="\t")
ds_new["P"] = fill_bgh_var(df_P, ds_new["S"], repeat=15)
ds_new["P_lambda"] = ds_new["S_lambda"]

In [ ]:
ds_new["u_lambda"].values[:] = 0
ds_new["u_lambda"][:, :, :nothern_edge, :] = (1 / (60 * 60 * 24)) * np_mask_u[:, :nothern_edge, :]
ds_new["u_lambda"] = ds_new["u_lambda"].astype(np.float32)

In [ ]:
ds_new["v_lambda"].values[:] = 0
ds_new["v_lambda"][:, :, :nothern_edge, :] = (1 / (60 * 60 * 24)) * np_mask_v[:, :nothern_edge, :]
ds_new["v_lambda"] = ds_new["v_lambda"].astype(np.float32)

Rivers

In [ ]:
def get_river(file_name):
    df_Barumsbassenget_DOM = pd.read_csv(
        os.path.join(Path.home(), "FjordsSim_data", "oslofjord", file_name), sep="\t"
    )
    df = df_Barumsbassenget_DOM
    arr = df.values[:, 2]
    arr = np.append(arr, arr[-1])
    arr = np.repeat(arr[:, None], 2, axis=1)
    return arr

Add a river in the Drammenfjord

In [ ]:
ds_new["S"].values[:, -2:, 186, 8] = 0
ds_new["S_lambda"].values[:, -2:, 186, 8] = 1 / (60 * 60)
ds_new["v"].values[:, -2:, 187, 8] = -0.5
ds_new["v_lambda"].values[:, -2:, 187, 8] = -2

Add a river in the Barumsbassenget

In [ ]:
ds_new["S"].values[:, -2:, 218, 43] = 0
ds_new["S_lambda"].values[:, -2:, 218, 43] = 1 / (60 * 60)
ds_new["u"].values[:, -2:, 218, 43] = 0.5
ds_new["u_lambda"].values[:, -2:, 218, 43] = 2

In [ ]:
ds_new["DOM"].values[:, -2:, 218, 43] = get_river("river_DON0_from_14_year_2021.txt")
ds_new["DOM_lambda"].values[:, -2:, 218, 43] = 1 / (60 * 60)

In [ ]:
ds_new["NUT"].values[:, -2:, 218, 43] = get_river("river_N3_n_from_14_year_2021.txt")
ds_new["NUT_lambda"].values[:, -2:, 218, 43] = 1 / (60 * 60)

Checkup

In [ ]:
ds_day = ds_new.sel(time="2024-02-29")
ds_day

In [ ]:
ds_day.S.isel(time=-1, Nz=-1).where(lambda x: x != 0).plot(vmin=-1, vmax=30, cmap=mpl.colormaps.get_cmap('Spectral'))

In [ ]:
ds_day.S_lambda.isel(time=-1, Nz=-1).where(lambda x: x != 0).plot(vmin=-1, vmax=30)

In [ ]:
ds_day.T.isel(time=-1, Nz=-1).where(lambda x: x != 0).plot(vmin=-1, vmax=5)

In [ ]:
ds_day.T_lambda.isel(time=-1, Nz=-1).where(lambda x: x != 0).plot()

In [ ]:
ds_day.u.isel(time=-1, Nz=-1).where(lambda x: x != 0).plot(vmin=-0.5, vmax=0.5)

In [ ]:
ds_day.u_lambda.isel(time=-1, Nz=-1).where(lambda x: x != 0).plot()

In [ ]:
ds_day.v.isel(time=-1, Nz=-1).where(lambda x: x != 0).plot(vmin=-0.5, vmax=0.5)

In [ ]:
ds_day.v_lambda.isel(time=-1, Nz=-1).where(lambda x: x != 0).plot()

In [ ]:
ds_grid.h.sel(lon=slice(10.2, 10.6), lat=slice(59.6, 59.9)).where(lambda x: x < 0).plot()

In [ ]:
ds_day.S.isel(time=-1, Nz=-1).sel(Nx=slice(10.2, 10.6), Ny=slice(59.6, 59.9)).where(lambda x: x != 0).plot(vmin=-1)

In [ ]:
ds_day.S_lambda.isel(time=-1, Nz=-1).sel(Nx=slice(10.2, 10.6), Ny=slice(59.6, 59.9)).where(lambda x: x != 0).plot()

In [ ]:
ds_day.u.isel(time=-1, Nz=-1).sel(Nx_faces=slice(10.2, 10.6), Ny=slice(59.6, 59.9)).where(
    lambda x: x != 0
).plot(vmin=-1, vmax=1)

In [ ]:
ds_day.u_lambda.isel(time=-1, Nz=-1).sel(Nx_faces=slice(10.2, 10.6), Ny=slice(59.6, 59.9)).where(
    lambda x: x != 0
).plot()

In [ ]:
ds_day.v_lambda.isel(time=-1, Nz=-1).sel(Nx=slice(10.2, 10.6), Ny_faces=slice(59.6, 59.9)).where(
    lambda x: x != 0
).plot()

In [ ]:
ds_new.to_netcdf(os.path.join(Path.home(), "FjordsSim_data", "oslofjord", "OF_inner_105to232_forcing_v2.nc"))